## imports:

In [25]:
%run given.ipynb
import datetime as dt
import json
import re
from types import SimpleNamespace


Salon backend ready: 6 seed bookings. NOW = 2026-10-19 10:00 (Monday)
22 public scenarios loaded.


### cancellation fee cents:

In [17]:
def cancellation_fee_cents(price_cents, start, now):
    """Fee in cents; raise ValueError if the appointment has already started."""
    remaining = start - now
    if remaining <= dt.timedelta(0):
        raise ValueError('the appointment has already started')
    if remaining >= dt.timedelta(hours=24):
        return 0
    if remaining >= dt.timedelta(hours=2):
        return price_cents // 2
    return price_cents

# class SalonAssistant:
#     def __init__(self, model, backend, clock):
#         self.model, self.backend, self.clock = model, backend, clock
#         self.sessions, self.audit = {}, []

#     def open_session(self, session_id, customer_id):
#         raise NotImplementedError('open_session')

#     def handle(self, session_id, user_text):
#         """Return the reply text for the customer."""
#         raise NotImplementedError('handle')

### check cancellation fee cents:

In [18]:
n = dt.datetime(2026, 10, 19, 10, 0)
H = lambda h, m=0: n + dt.timedelta(hours=h, minutes=m)

assert cancellation_fee_cents(3500, H(24), n) == 0   
assert cancellation_fee_cents(3500, H(23, 59), n) == 1750
assert cancellation_fee_cents(3501, H(5), n) == 1750    
assert cancellation_fee_cents(3500, H(2), n) == 1750     
assert cancellation_fee_cents(3500, H(1, 59), n) == 3500

for bad in (n, H(-1)):
    try:
        cancellation_fee_cents(3500, bad, n)
        raise SystemExit("it should be value eror !")
    except ValueError:
        pass
print("fee OK")

fee OK


### interpert reply :

In [19]:
_YES = (
    r"(?:yes|yeah|yep|yup|y|ok|okay|sure|absolutely|definitely|perfect|great|alright|fine|"
    r"confirm|confirmed|correct|go ahead|that works(?: for me)?|(?:that )?sounds good|"
    r"do it|please do|go for it|book it)"
)
_YES_TAIL = r"(?: (?:yes|please|thanks|thank you|do it|do|go ahead|perfect|great|sure|ok|okay))*"
_NO = r"(?:no|nope|nah|stop|not yet|never mind|nevermind|don't|dont|do not|please don't|please dont)"
_NO_TAIL = r"(?: (?:thanks|thank you|please|it|that|book it|cancel it|do it|now))*"
YES_RE = re.compile(rf"^{_YES}{_YES_TAIL}$")
NO_RE  = re.compile(rf"^{_NO}{_NO_TAIL}$")

def interpret_reply(text):
    """Return 'yes', 'no' or 'unclear'. Conservative: when in doubt, 'unclear'."""
    if not isinstance(text, str) or '?' in text:
        return 'unclear'

    s = text.lower().replace('’', "'")
    s = re.sub(r"[^a-z0-9' ]+", ' ', s)
    s = ' '.join(s.split()) 

    if YES_RE.match(s):
        return 'yes'
    if NO_RE.match(s):
        return 'no'
    return 'unclear'


### tool schema:

In [20]:
_SERVICE_IDS = ['cut', 'beard', 'wash_style', 'color', 'highlights']
_STYLIST_IDS = ['mara', 'jon', 'lea']
_START_PATTERN = r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}$'

def _tool(name, description, properties=None, required=None):
    return {'type': 'function', 'function': {
        'name': name, 'description': description,
        'parameters': {'type': 'object', 'properties': properties or {},
                       'required': required or [], 'additionalProperties': False}}}

_SERVICE = {'type': 'string', 'enum': _SERVICE_IDS, 'description': 'Service id'}
_STYLIST = {'type': 'string', 'enum': _STYLIST_IDS, 'description': 'Stylist id'}
_BOOKING = {'type': 'string', 'pattern': r'^B-\d+$', 'description': 'Booking id such as B-2'}
_START = {'type': 'string', 'pattern': _START_PATTERN,
          'description': "Local time 'YYYY-MM-DD HH:MM' on a 30-minute mark"}

TOOL_SCHEMAS = [
    _tool('list_services', 'List the salon services with durations and prices.'),
    _tool('find_slots', 'Find free appointment slots for a service on one day.',
          {'service_id': _SERVICE,
           'date': {'type': 'string', 'pattern': r'^\d{4}-\d{2}-\d{2}$', 'description': "Date 'YYYY-MM-DD'"},
           'stylist_id': _STYLIST}, ['service_id', 'date']),
    _tool('get_my_bookings', "List the logged-in customer's own bookings."),
    _tool('book_appointment', 'PROPOSE a booking. Nothing is booked until the customer confirms in their next message.',
          {'service_id': _SERVICE, 'stylist_id': _STYLIST, 'start': _START},
          ['service_id', 'stylist_id', 'start']),
    _tool('reschedule_appointment', "PROPOSE moving one of the customer's bookings to a new start time (confirmation needed).",
          {'booking_id': _BOOKING, 'new_start': _START}, ['booking_id', 'new_start']),
    _tool('cancel_appointment', "PROPOSE cancelling one of the customer's bookings (confirmation needed; a fee may apply).",
          {'booking_id': _BOOKING}, ['booking_id']),
    _tool('handoff_to_human', 'Pass the case to a human team member.',
          {'reason': {'type': 'string', 'enum': ['customer_request','policy_exception', 'tool_failures', 'other'],
                      'description': 'Why a human is needed'}}, ['reason']),
]

### check p22 & p05:

In [21]:
sc_fee_table(None); print("p22 fee table ok")
sc_reply_table(None); print("p05 reply table ok")

sc_schemas(SimpleNamespace(model=SimpleNamespace(tools_seen=TOOL_SCHEMAS)))
print("p21 schemas ok ")

p22 fee table ok
p05 reply table ok
p21 schemas ok 


### SalonAssistant:

In [23]:
MAX_MODEL_CALLS, MAX_TOOL_CALLS, MAX_MESSAGES = 6, 8, 24

_EMAIL_RE = re.compile(r'[\w.+-]+@[\w-]+(?:\.[\w-]+)+')
_DATE_OR_PHONE_RE = re.compile(
    r'(?P<date>\d{4}-\d{2}-\d{2}(?: \d{2}:\d{2})?)'
    r'|(?P<phone>\+?\(?\d{1,4}\)?[\s.-]?\d{2,4}[\s.-]?\d{4,})'
)

def _redact(text):
    """Hide e-mails and phone numbers (but keep ISO dates/times readable)."""
    text = _EMAIL_RE.sub('[email]', str(text))
    def repl(m):
        if m.group('date'):
            return m.group(0)
        return '[phone]' if sum(c.isdigit() for c in m.group(0)) >= 7 else m.group(0)
    return _DATE_OR_PHONE_RE.sub(repl, text)

def _euro(cents):
    return f'€{cents // 100}.{cents % 100:02d}'

_SCHEMA_BY_NAME = {t['function']['name']: t['function']['parameters'] for t in TOOL_SCHEMAS}

SLOT_MIN = 30
def parse_dt(s): return dt.datetime.strptime(s, '%Y-%m-%d %H:%M') 

def _validate_args(name, args):
    """Return an error message, or None when the arguments fit the tool's schema."""
    schema = _SCHEMA_BY_NAME[name]
    props = schema['properties']
    extra = set(args) - set(props)
    if extra:
        return f'unexpected arguments: {sorted(extra)}'
    for key in schema['required']:
        if key not in args:
            return f'missing argument: {key}'
    for key, value in args.items():
        spec = props[key]
        if not isinstance(value, str):
            return f'{key} must be a string'
        if 'enum' in spec and value not in spec['enum']:
            return f'{key} must be one of {spec["enum"]}'
        if 'pattern' in spec and not re.match(spec['pattern'], value):
            return f'{key} has the wrong format'
    try:
        if 'date' in args:
            dt.datetime.strptime(args['date'], '%Y-%m-%d')
        for key in ('start', 'new_start'):
            if key in args and parse_dt(args[key]).minute % SLOT_MIN:
                return f'{key} must be on a 30-minute mark'
    except ValueError:
        return 'the date/time is not a real calendar value'
    return None

def _result(status, error_type=None, data=None, attempts=1):
    return {'status': status, 'error_type': error_type, 'data': data, 'attempts': attempts}

In [33]:
class SalonAssistant:
    def __init__(self, model, backend, clock):
        self.model, self.backend, self.clock = model, backend, clock
        self.sessions, self.audit = {}, []
    def open_session(self, session_id, customer_id):
        self.sessions[session_id] = {
            'id': session_id, 'customer':customer_id,   
            'messages': [],                             
            'pending': None,                           
            'turn': 0, 'seq': 0, 'seen_ids': set(),
        }

    def handle(self, session_id, user_text):
        """Return the reply text for the customer."""
        st = self.sessions[session_id]
        st['turn'] += 1
        turn = {'model_calls': 0, 'tool_calls': 0}
        user_text = user_text if isinstance(user_text, str) else str(user_text)
        self._audit(st, None, '(user)', None, None, 0, user_text)

        pending, st['pending'] = st['pending'], None      
        st['messages'].append({'role': 'user', 'content': user_text})
        if pending is not None and interpret_reply(user_text) == 'yes':
            self._execute_pending(st, turn, pending)
        return self._run_model(st, turn)


    def _system_prompt(self):
        now = self.clock()
        return (
            "You are Luna, the booking assistant of Studio Linden (a hair salon).\n"
            f"Now: {fmt(now)} ({now.strftime('%A')}). Times are local salon time.\n"
            "Rules:\n"
            "- Use the tools for every fact: services, prices, free slots and the customer's bookings.\n"
            "- book_appointment, reschedule_appointment and cancel_appointment only PROPOSE a change. "
            "After a proposal, ask the customer to confirm; nothing happens until they say yes in their next message.\n"
            "- Never say a booking was made, moved or cancelled unless a tool result says so.\n"
            "- Never state a time, price or booking id that did not come from a tool result or from the customer.\n"
            "- You only know the logged-in customer; never discuss anyone else's bookings.\n"
            "- If the customer wants a person, use handoff_to_human."
        )

    def _build_messages(self, st):
        msgs = st['messages']
        starts = [i for i, m in enumerate(msgs) if m['role'] == 'user']
        keep_from = starts[-1] if starts else 0         
        for s in reversed(starts[:-1]):               
            if len(msgs) - s <= MAX_MESSAGES - 1:
                keep_from = s
            else:
                break
        return [{'role': 'system', 'content': self._system_prompt()}] + msgs[keep_from:]

    def _finish(self, st, text):
        st['messages'].append({'role': 'assistant', 'content': text})
        return text

    def _run_model(self, st, turn):
        while True:
            if turn['model_calls'] >= MAX_MODEL_CALLS:
                return self._finish(st, SAFE_FALLBACK)
            turn['model_calls'] += 1
            try:
                resp = self.model(self._build_messages(st), TOOL_SCHEMAS)
            except Exception:
                return self._finish(st, SAFE_FALLBACK)

            if isinstance(resp, dict) and resp.get('tool_calls') is not None:
                tool_calls = resp['tool_calls']
                if not self._valid_batch(st, turn, tool_calls) or turn['model_calls'] >= MAX_MODEL_CALLS:
                    return self._finish(st, SAFE_FALLBACK)
                self._run_tool_calls(st, turn, tool_calls)
                continue
            if isinstance(resp, dict) and isinstance(resp.get('final'), str):
                return self._finish(st, resp['final'])
            return self._finish(st, SAFE_FALLBACK)
    def _valid_batch(self, st, turn, tool_calls):
        """The whole batch is rejected (nothing runs) if its structure is wrong."""
        if not isinstance(tool_calls, list) or not tool_calls:
            return False
        if turn['tool_calls'] + len(tool_calls) > MAX_TOOL_CALLS:
            return False
        ids = []
        for tc in tool_calls:
            if not isinstance(tc, dict):
                return False
            if not isinstance(tc.get('id'), str) or not tc['id'] or not isinstance(tc.get('name'), str):
                return False
            if not isinstance(tc.get('arguments'), (dict, str)):
                return False
            ids.append(tc['id'])
        return len(set(ids)) == len(ids) and not (set(ids) & st['seen_ids'])

    def _run_tool_calls(self, st, turn, tool_calls):
        calls = []
        for tc in tool_calls:
            args = tc['arguments'] if isinstance(tc['arguments'], str) else json.dumps(tc['arguments'])
            calls.append({'id': tc['id'], 'type': 'function', 'function': {'name': tc['name'], 'arguments': args}})
            st['seen_ids'].add(tc['id'])
        st['messages'].append({'role': 'assistant', 'content': None, 'tool_calls': calls})
        for tc in tool_calls:
            turn['tool_calls'] += 1
            res = self._run_one(st, tc['name'], tc['arguments'])
            self._record(st, tc['id'], tc['name'], res)

    def _record(self, st, call_id, tool, res):
        """Show the result to the model and write the audit event."""
        obs = {'call_id': call_id, 'tool': tool, 'status': res['status']}
        if res['error_type']:
            obs['error_type'] = res['error_type']
        obs['data'] = res['data']
        st['messages'].append({'role': 'tool', 'tool_call_id': call_id, 'content': json.dumps(obs, ensure_ascii=False)})
        preview = '' if res['data'] is None else json.dumps(res['data'], ensure_ascii=False)
        self._audit(st, call_id, tool, res['status'], res['error_type'], res['attempts'], preview)

    def _audit(self, st, call_id, tool, status, error_type, attempts, preview):
        self.audit.append({'session': st['id'], 'turn': st['turn'], 'call_id': call_id, 'tool': tool,
                           'status': status, 'error_type': error_type, 'attempts': attempts,
                           'preview': _redact(preview)[:80]})

    def _run_one(self, st, name, arguments):
        if name not in _SCHEMA_BY_NAME:
            return _result('error', 'unknown_tool', attempts=0)
        args = arguments
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except ValueError:
                return _result('error', 'parse_error', attempts=0)
        if not isinstance(args, dict):
            return _result('error', 'validation_error', {'detail': 'arguments must be an object'}, 0)
        problem = _validate_args(name, args)
        if problem:
            return _result('error', 'validation_error', {'detail': problem}, 0)
        handler = {'list_services': self._t_list_services, 'find_slots': self._t_find_slots,
                   'get_my_bookings': self._t_get_my_bookings, 'book_appointment': self._t_book}.get(name)
        if handler is None:    
            return _result('error', 'not_implemented', attempts=0)
        try:
            return handler(st, args)
        except Exception as exc:  
            return _result('error', 'unexpected_error', {'detail': type(exc).__name__})

    def _t_list_services(self, st, args):
        services = self.backend.list_services()
        data = [{'service_id': sid, 'name': s['name'], 'minutes': s['minutes'],
                 'price_cents': s['price_cents'], 'price': _euro(s['price_cents'])} for sid, s in services.items()]
        return _result('ok', data=data)

    def _t_find_slots(self, st, args):        
        slots = self.backend.free_slots(args['service_id'], args['date'], args.get('stylist_id'))        
        return _result('ok' if slots else 'not_found', data=slots)

    def _t_get_my_bookings(self, st, args):        
        rows = self.backend.bookings_for(st['customer'])       
        data = [{k: b[k] for k in ('booking_id', 'service_id', 'stylist_id', 'start', 'status', 'reschedules')} for b in rows]       
        return _result('ok', data=data)

    def _alternatives(self, service_id, date, wanted_start, stylist_id=None, limit=3):
        slots = self.backend.free_slots(service_id, date, stylist_id)
        want = parse_dt(wanted_start)
        slots.sort(key=lambda r: (abs((parse_dt(r['start']) - want).total_seconds()), r['start'], r['stylist_id']))
        return slots[:limit]

    def _t_book(self, st, args):
        if st['pending'] is not None:
            return _result('blocked', 'pending_exists', {'detail': 'another change is already waiting for confirmation'}, 0)
        service_id, stylist_id, start = args['service_id'], args['stylist_id'], args['start']
        service, stylist = SERVICES[service_id], STYLISTS[stylist_id]
        if service_id not in stylist['services']:
            return _result('blocked', 'stylist_not_offering', {'detail': f"{stylist['name']} does not offer {service['name']}"}, 0)

        mine = [b for b in self.backend.bookings_for(st['customer']) if b['status'] == 'active']
        if len(mine) >= MAX_ACTIVE:
            return _result('blocked', 'limit_reached', {'detail': f'at most {MAX_ACTIVE} active bookings are allowed'},0)
        new_s = parse_dt(start)
        new_e = new_s + dt.timedelta(minutes=service['minutes'])
        for b in mine:
            s = parse_dt(b['start'])
            e = s + dt.timedelta(minutes=SERVICES[b['service_id']]['minutes'])
            if new_s < e and s < new_e:
                return _result('blocked', 'overlap', {'booking_id': b['booking_id'], 'detail': 'overlaps another booking of the customer'},0)

        date = start[:10]
        free = self.backend.free_slots(service_id, date)
        if {'stylist_id': stylist_id, 'start': start} not in free:
            alts = self._alternatives(service_id, date, start)
            return _result('blocked', 'slot_unavailable', {'alternatives': alts})

        summary = (f"Book {service['name']} with {stylist['name']} on {date} at {start[11:]} "
                   f"for {_euro(service['price_cents'])}.")
        st['pending'] = {'tool': 'book_appointment', 'args': dict(args), 'summary': summary}
        return _result('needs_confirmation', data={'summary': summary})

    def _execute_pending(self, st, turn, pending):
        """The customer said yes: run the stored action ourselves and show the model a synthetic tool call."""
        st['seq'] += 1
        call_id = f"exec-{st['turn']}-{st['seq']}"
        while call_id in st['seen_ids']:
            call_id += 'x'
        st['seen_ids'].add(call_id)
        turn['tool_calls'] += 1
        name, args = pending['tool'], pending['args']
        st['messages'].append({'role': 'assistant', 'content': None, 'tool_calls': [
            {'id': call_id, 'type': 'function', 'function': {'name': name, 'arguments': json.dumps(args)}}]})
        request_key = f"{st['id']}:{st['turn']}:{st['seq']}"        
        try:
            if name == 'book_appointment':
                done = self.backend.book(st['customer'], args['service_id'], args['stylist_id'], args['start'], request_key)
                data = {k: done[k] for k in ('booking_id', 'service_id', 'stylist_id', 'start', 'status')}
                res = _result('ok', data=data)
            else:
                res = _result('error', 'not_implemented', attempts=0)
        except Exception as exc:
            res = _result('error', 'unexpected_error', {'detail': type(exc).__name__})
            
        self._record(st, call_id, name, res)   
      

### checking :

In [35]:
for sid in ('P01','P02','P03'):
    trace(sid , SalonAssistant)

=== P01 Greeting needs no tools ===

[turn 1 ('Hi!')]
  USER : Hi!
  REPLY: Hello! How can I help you today?

RESULT: PASS
=== P02 Browse: two independent reads in one turn ===

[turn 1 ("What do you offer, and what'")]
  USER : What do you offer, and what's free for a haircut on Tuesday?
  REPLY: We offer Haircut (€35.00), Beard trim, Wash & style, Colour and Highlights. Tuesday haircut slots start at 09:00 and 09:30.

RESULT: PASS
=== P03 Book: propose, then confirm in a LATER turn ===

[turn 1 ('Please book me a haircut wit')]
  USER : Please book me a haircut with Mara tomorrow at 11:00
  REPLY: Here is what I will book: Haircut with Mara on 2026-10-20 at 11:00 for €35.00. Shall I go ahead?

[turn 2 ('yes please')]
  USER : yes please
  REPLY: Done! Your haircut with Mara is booked for 2026-10-20 at 11:00 (B-7).

RESULT: PASS
